IPL DATASET

Problem Statement :- IPL 2025 Player Stats Analysis and Prediction

To predict IPL match outcomes by analyzing the underlying factors within the high-dimensional match dataset. After preprocessing the raw data,we apply Principal Component Analysis (PCA) to reduce the numerous features into a concise set of latent components.Then we Visualize these components to identify and understand hidden patterns and team behaviors. Finally,we use this reduced feature space to build an efficient and accurate classification model.

Importing the dataset of ipl matches.csv and all the libraries

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from scipy import stats

df = pd.read_csv("matches.csv")
print("Dataset loaded:", df.shape)
print(f"Number of rows: {df.shape[0]}")
print(f"Number of columns: {df.shape[1]}")
print("\nColumn names:")
print(df.columns.tolist())
print(df.head())

Steps we’ll cover:

Data Cleaning – Handle missing values, duplicates, and invalid entries.

Data Processing – Convert types (e.g., date), normalize text.

Data Transformation – Encode categorical variables.

Feature Engineering – Create new useful features.

Feature Selection – Drop irrelevant or redundant features.

Handling & Balancing Data – Address class imbalance (target column: winner).

Splitting Data – Train/Test split.

In [ ]:
# Load dataset
df = pd.read_csv("matches.csv")

# Data Cleaning
# Convert 'date' column to datetime
df['date'] = pd.to_datetime(df['date'], errors='coerce')

# Check missing values
print("Missing values per column:")
print(df.isnull().sum())

# Drop rows with invalid dates
df = df.dropna(subset=['date'])

# Drop duplicate rows
df = df.drop_duplicates()

# Fix typos (example corrections for team/city names if needed)
df['city'] = df['city'].replace({'Bengaluru': 'Bangalore'})
df['team1'] = df['team1'].replace({'Delhi Daredevils': 'Delhi Capitals'})
df['team2'] = df['team2'].replace({'Delhi Daredevils': 'Delhi Capitals'})


# Aggregation

# Take latest match info for each winner (similar to last record per State)
df_latest = df.groupby('winner').last().reset_index()

# Derived Feature: Matches Played (id - season just for example; adjust meaningfully)
# Removed this line as subtracting 'season' (string) from 'id' (int) is not a valid operation
# and doesn't seem relevant to the problem statement.
# df_latest['MatchesPlayed'] = df_latest['id'] - df_latest['season']

# Select relevant numeric features
# Removed 'MatchesPlayed' and 'id' from the features list as 'id' is not a meaningful numerical feature
features = ['result_margin', 'target_runs', 'target_overs']
X = df_latest[features].fillna(0) # Fill potential missing values with 0 for numeric features

# Feature Scaling

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Encode categorical column: Winner
le = LabelEncoder()
df_latest['Winner_encoded'] = le.fit_transform(df_latest['winner'].astype(str))


# Final Processed Data


print("Processed data (latest per winner):")
print(df_latest.head())

print("\nScaled numeric features (first 5 rows):")
display(pd.DataFrame(X_scaled, columns=features).head())

Principal Component Analysis

PCA (Principal Component Analysis) is a dimensionality reduction technique used in data analysis and machine learning. It helps you to reduce the number of features in a dataset while keeping the most important information. It changes your original features into new features these new features don’t overlap with each other and the first few keep most of the important differences found in the original data.

Steps to apply the PCA -

1. Import Necessary Libraries
2. Load the Dataset
3. Standardizing the Data
4. Check the Correlation Between Features (Without PCA)
5. Applying PCA
6. Check Correlation Between Features (After PCA)

In [ ]:
# Step 1: Load the dataset and preprocess
print("--- Step 1: Loading and Preprocessing Dataset ---")
# Load dataset
df = pd.read_csv('matches.csv')

# Data Cleaning
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df = df.dropna(subset=['date'])
df = df.drop_duplicates()

# Fix typos in 'city' and teams if needed
df['city'] = df['city'].replace({'Bengaluru': 'Bangalore'})
df['team1'] = df['team1'].replace({'Delhi Daredevils': 'Delhi Capitals'})
df['team2'] = df['team2'].replace({'Delhi Daredevils': 'Delhi Capitals'})

# Aggregate - take latest match info per winner
df_latest = df.groupby('winner').last().reset_index()

# Derived Feature: MatchesPlayed (example feature using id and season)
# Removed this line as subtracting 'season' (string) from 'id' (int) is not a valid operation
# and doesn't seem relevant to the problem statement.
# df_latest['MatchesPlayed'] = df_latest['id'] - df_latest['season']


print("Processed data (latest per winner):")
display(df_latest.head())
print("----------------------------------------------")

# Select relevant numeric features for PCA
# Removed 'MatchesPlayed' and 'id' from the features list as 'id' is not a meaningful numerical feature
features = ['result_margin', 'target_runs', 'target_overs']


# Step 2: Standardize/Normalize the features
print("\n--- Step 2: Standardizing Features ---")
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_latest[features])
print("Scaled features (first 5 rows):")
print(X_scaled[:5])
print("------------------------------------")

# Step 3: Correlation Matrix BEFORE PCA
print("\n--- Step 3: Correlation Matrix BEFORE PCA ---")
X_scaled_df = pd.DataFrame(X_scaled, columns=features)
correlation_matrix_before = X_scaled_df.corr()
print("Correlation matrix:")
display(correlation_matrix_before)
plt.figure(figsize=(6, 4))
sns.heatmap(correlation_matrix_before, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Matrix of Features Before PCA')
plt.show()
print("------------------------------------------")

# Step 4: Applying PCA
print("\n--- Step 4: Applying PCA ---")
pca = PCA()
X_pca = pca.fit_transform(X_scaled)
explained_variance_ratio = pca.explained_variance_ratio_
cumulative_explained_variance = explained_variance_ratio.cumsum()
print("Explained variance ratio by each component:")
print(explained_variance_ratio)
print("\nCumulative explained variance:")
print(cumulative_explained_variance)

# Create a DataFrame with PCA components
pca_df = pd.DataFrame(X_pca, columns=[f'PC{i+1}' for i in range(X_pca.shape[1])])
pca_df['winner'] = df_latest['winner'].values  # Add winner for context
print("\nPCA components head:")
display(pca_df.head())
print("--------------------------")

# Step 5: Correlation Matrix AFTER PCA
print("\n--- Step 5: Correlation Matrix AFTER PCA (Principal Components) ---")
pca_components_df = pd.DataFrame(X_pca, columns=[f'PC{i+1}' for i in range(X_pca.shape[1])])
correlation_matrix_after = pca_components_df.corr()
print("Correlation matrix:")
display(correlation_matrix_after)
plt.figure(figsize=(6, 4))
sns.heatmap(correlation_matrix_after, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Matrix of Principal Components After PCA')
plt.show()
print("-------------------------------------------")

print("\n--- PCA Analysis Complete ---")

Data Visualization

Data visualization uses charts, graphs and maps to present information clearly and simply. It turns complex data into visuals that are easy to understand.

With large amounts of data in every industry, visualization helps spot patterns and trends quickly, leading to faster and smarter decisions.

Their are some methods like Heatmaps,Bar charts,Box plots,Scatter Plots,Violin plots,Histogram etc.

**1. Heatmap** :-
Heatmap data visualization is a powerful tool used to represent numerical data graphically, where values are depicted using colors. This method is particularly effective for identifying patterns, trends, and anomalies within large datasets.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

# Heat Map
print("--- Generating Heat Map ---")

# Numerical columns in matches dataset for correlation analysis
# Removed 'id' and 'season' as they are not suitable for correlation
numerical_cols_for_heatmap = ['result_margin', 'target_runs', 'target_overs']

# Calculate correlation matrix
correlation_matrix = df_latest[numerical_cols_for_heatmap].corr()

# Plot heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Heatmap of Feature Correlation (Matches Dataset)')
plt.show()

print("-------------------------")

**2. Histogram** :-
A histogram is a type of graphical representation used in statistics to show the distribution of numerical data.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Histogram
print("--- Generating Histograms ---")

# Example Histograms for numerical columns in matches dataset
# Removed 'id', 'season' and 'MatchesPlayed' as they are not suitable for histograms
numerical_cols_for_hist = ['result_margin', 'target_runs', 'target_overs']

df_latest[numerical_cols_for_hist].hist(bins=15, figsize=(10, 6))
plt.suptitle('Histograms of Numerical Features (Matches Dataset)', y=1.02)
plt.tight_layout(rect=[0, 0, 1, 0.95])  # Adjust layout to prevent title overlap
plt.show()

print("---------------------------")

**3. Pair Plot** :- A pair plot,visualizes the pairwise relationships and individual distributions of multiple numerical variables in a dataset

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

# Load the data (assuming df is the DataFrame)
# df = pd.read_csv('matches.csv')

# Pair Plot
print("--- Generating Pair Plot ---")

# Select a subset of relevant numerical columns for the pair plot
numerical_cols_for_pairplot = ['result_margin', 'target_runs', 'target_overs']

# Drop rows with NaN values in the selected columns for accurate plotting
df_cleaned = df.dropna(subset=numerical_cols_for_pairplot)

# Generate the pair plot
sns.pairplot(df_cleaned[numerical_cols_for_pairplot])
plt.suptitle('Pair Plot of Numerical Match Features', y=1.02) # Add a title to the plot
plt.savefig('pair_plot_numerical_features.png')
# plt.show() is replaced with plt.savefig for the environment

print("--------------------------")

**4. Bar chart** :- A bar plot uses rectangular bars to represent data categories, with bar
length or height proportional to their values.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Load the matches.csv file
df = pd.read_csv('matches.csv')

# Data preparation: Calculate the number of wins for each team and sort
wins_by_team = df['winner'].value_counts().reset_index()
wins_by_team.columns = ['Team', 'Wins']
wins_by_team = wins_by_team.sort_values(by='Wins', ascending=False)

# Bar Chart
print("--- Generating Bar Chart (Wins per Team) ---")

# Plotting the data
plt.figure(figsize=(12, 6))
# We use the prepared DataFrame `wins_by_team`
sns.barplot(x='Team', y='Wins', data=wins_by_team, palette='viridis')
plt.title('Number of Matches Won by Each Team (All Seasons)', fontsize=16)
plt.xlabel('Team', fontsize=12)
plt.ylabel('Number of Wins', fontsize=12)
plt.xticks(rotation=45, ha='right') # Rotate team names for readability
plt.tight_layout() # Adjust layout
plt.savefig('wins_per_team_bar_chart.png')
# plt.show()

print("-----------------------------")

**5. Violin Plot** :- Violin plots are used to visualize the distribution of a single continuous variable.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Load the matches.csv file
df = pd.read_csv('matches.csv')

# Columns for the violin plot
categorical_col = 'result'
numerical_col = 'result_margin'

# Data Preparation: Filter and clean data for the plot
# 1. Filter out non-run/non-wicket results (tie, no result, NA values)
df_cleaned_violin = df[df[categorical_col].isin(['runs', 'wickets'])].copy()
# 2. Drop rows with NaN in the numerical column
df_cleaned_violin.dropna(subset=[numerical_col], inplace=True)

# Violin Plot
print("--- Generating Violin Plot (Distribution of Result Margin by Result Type) ---")

# Plotting the data
plt.figure(figsize=(10, 6))
sns.violinplot(x=categorical_col, y=numerical_col, data=df_cleaned_violin, palette={'runs': 'salmon', 'wickets': 'lightskyblue'})
plt.title('Distribution of Result Margin by Result Type', fontsize=16)
plt.xlabel('Result Type', fontsize=12)
plt.ylabel('Result Margin', fontsize=12)
plt.tight_layout() # Adjust layout
plt.savefig('violin_plot_result_margin.png')
# plt.show()

print("----------------------------")